# The Track

> A Track is the durable record of an Op or Cog execution.
>
> — [@oliphant2026, §5.3]

`track/run-001.trig` records the escalated run as PROV-O [@provo2013]
with EARL outcomes [@earl2017]: automatic Guard results (with an honest
cantTell among the outcomes), fired Gates raising obligations, and an
`earl:manual` approval by a named person that discharges both
obligations and generates the post-review case state
(an action that discharges an obligation must also mutate state, a
ruling logged as GAP-02). This run's discharger happens to be human, as
the two gates that fired require; the shapes demand a declared mode and
a named accountable party, not humanness (GAP-09 in the appendix).

run-001 is itself fabricated example data (the record's own header says
so): a constructed execution that exercises the shapes and queries. Its
approver is an invented person, named only because a discharging action
must name an accountable party.

SHACL shapes [@shacl2017] derived from the manifest's own
`track.include` list check the record. The cell states what the record
contains, then what the shapes conclude:

In [1]:
import sys; sys.path[:0] = [".", ".."]  # the repo root, from either cwd
import exhibits
exhibits.check_run_001_shapes()

data checked: track/run-001.trig
  assertions          : 4 automatic, 1 manual (outcomes: 1 cantTell, 2 failed, 2 passed)
  gate decisions      : 2
  obligations raised  : obligation-human-approval, obligation-human-review
  discharging actions :
    approval-01 discharges obligation-human-approval
    approval-01 discharges obligation-human-review
conforms: True


The same shapes can refuse. The counterexample record raises the same
two obligations and contains no discharging action; that absence is the
only difference in the data, so the refusal is checkable by eye, one
violation per undischarged obligation.

One acknowledged judgment call sits in these shapes: retention of
`final_output` is conditional on the aggregate outcome (an executed run
must retain it, a noOp run must not). That knowingly departs from the
pinned draft's unconditional include list, and the departure is stated
in the shape messages themselves (GAP-07 in the appendix).

In [2]:
exhibits.refuse_missing_approval_track()

data checked: counterexamples/track-missing-approval.trig
  assertions          : 4 automatic (outcomes: 1 cantTell, 2 failed, 1 passed)
  gate decisions      : 2
  obligations raised  : obligation-human-approval, obligation-human-review
  discharging actions : (none)
conforms: False (2 violations; focus nodes equal the undischarged obligations: True)

  violation 1 — focus node: obligation-human-approval
    Obligation raised but not discharged: no discharging action
    recorded (section 5.6 lists human_approvals in the Track; only the
    escalation target's actor is unspecified — adjudicated: GAP-09).

  violation 2 — focus node: obligation-human-review
    Obligation raised but not discharged: no discharging action
    recorded (section 5.6 lists human_approvals in the Track; only the
    escalation target's actor is unspecified — adjudicated: GAP-09).


> The Track is not just a log file. It is a structured accountability
> artifact.
>
> — [@oliphant2026, §5.3]

Section 5.3 names five purposes a Track serves: auditability,
governance, learning, debugging, and trust. On this substrate each one
is a query [@sparql2013] over the run graph, not a reading exercise.

## Auditability

> Auditability — an organization can reconstruct why a decision was
> made.
>
> — [@oliphant2026, §5.3]

In [3]:
exhibits.show_query("auditability.rq");

gate,condition,guardFinding,obliged,evidence,approver,rationale
GATE-01,confidence < 0.80,extraction confidence 0.71 is below the 0.80 gate threshold,human review required,"Invoice batch 2026-08 (vendor V-2214), 37 invoices",J. Rivera (accounts-payable analyst),Reviewed the three lowest-confidence extractions against the invoice PDFs; amounts and dates check out. The risk score is driven by the 40 percent month-over-month invoice volume increase; flagging for procurement follow-up is warranted.
GATE-01,confidence < 0.80,extraction confidence 0.71 is below the 0.80 gate threshold,human review required,Vendor risk score for V-2214: high,J. Rivera (accounts-payable analyst),Reviewed the three lowest-confidence extractions against the invoice PDFs; amounts and dates check out. The risk score is driven by the 40 percent month-over-month invoice volume increase; flagging for procurement follow-up is warranted.
GATE-04,vendor_risk == high,vendor-risk-cog scored vendor V-2214 as high,human approval required,"Invoice batch 2026-08 (vendor V-2214), 37 invoices",J. Rivera (accounts-payable analyst),Reviewed the three lowest-confidence extractions against the invoice PDFs; amounts and dates check out. The risk score is driven by the 40 percent month-over-month invoice volume increase; flagging for procurement follow-up is warranted.
GATE-04,vendor_risk == high,vendor-risk-cog scored vendor V-2214 as high,human approval required,Vendor risk score for V-2214: high,J. Rivera (accounts-payable analyst),Reviewed the three lowest-confidence extractions against the invoice PDFs; amounts and dates check out. The risk score is driven by the 40 percent month-over-month invoice volume increase; flagging for procurement follow-up is warranted.


## Governance

> Governance — compliance teams can verify that required procedures
> were followed.
>
> — [@oliphant2026, §5.3]

Rows are violations; empty means every obligation raised by a fired
Gate was discharged by a recorded action with a named accountable
party. The same query catches the counterexample:

In [4]:
exhibits.compare_governance()

(no rows)
violations on run-001: 0
violations on the counterexample: 2


obligation,obliged,gate
run:obligation-human-review,human review required,GATE-01
run:obligation-human-approval,human approval required,GATE-04


## Trust

> Trust — users and customers can see that AI work was not merely
> generated, but validated.
>
> — [@oliphant2026, §5.3]

The full outcome distribution, automatic and manual, with cantTell
visible rather than absorbed:

In [5]:
exhibits.show_query("trust.rq");

assertion,label,mode,outcome
run:guard-confidence,confidence-guard (in_flight),earl:automatic,earl:failed
run:guard-schema,schema-guard (post_run),earl:automatic,earl:passed
run:guard-source-grounding,source-grounding-guard (post_run),earl:automatic,earl:cantTell
run:guard-vendor-risk-reading,vendor risk reading (in_flight),earl:automatic,earl:failed
run:approval-01,human review and approval of the vendor flag,earl:manual,earl:passed


## Learning

> Learning — corrections and human reviews become signals for improving
> Frames, Cogs, Ops, and Guards.
>
> — [@oliphant2026, §5.3]

Every human review with its rationale, and every automatic guard
finding that did not pass, is an improvement signal this run left
behind:

In [6]:
exhibits.show_query("learning.rq");

signal,kind,detail
run:guard-confidence,guard finding,extraction confidence 0.71 is below the 0.80 gate threshold
run:guard-source-grounding,guard finding,the cited invoice passage is ambiguous about the delivery date; the claim can be neither grounded nor refuted from the approved sources
run:guard-vendor-risk-reading,guard finding,vendor-risk-cog scored vendor V-2214 as high
run:approval-01,human review,Reviewed the three lowest-confidence extractions against the invoice PDFs; amounts and dates check out. The risk score is driven by the 40 percent month-over-month invoice volume increase; flagging for procurement follow-up is warranted.


## Debugging

> Debugging — developers can understand why an Op failed or behaved
> unexpectedly.
>
> — [@oliphant2026, §5.3]

For each guard finding that did not pass: the recorded explanation
and, where a gate evaluated an oracle-provided value on the strength
of that finding, the cited call that produced the value:

In [7]:
exhibits.show_query("debugging.rq");

guard,outcome,info,service,responsePayload
run:guard-confidence,earl:failed,extraction confidence 0.71 is below the 0.80 gate threshold,urn:example:service:extraction-confidence,"{""confidence"":0.71}"
run:guard-source-grounding,earl:cantTell,the cited invoice passage is ambiguous about the delivery date; the claim can be neither grounded nor refuted from the approved sources,,
run:guard-vendor-risk-reading,earl:failed,vendor-risk-cog scored vendor V-2214 as high,urn:example:service:vendor-risk-cog,"{""vendor_risk"":""high""}"


## The interface: where every value came from

This purpose is not on the paper's list; it is a consequence of a
ruling logged as GAP-05. The policy applies to oracle-provided values
and is never their provider, so for each variable a gate evaluated the
Track must cite the call: what service, what payload was sent, what
response code came back, and the response that carried the value. The
numerical precision lives inside the oracles or in documented threshold
rules; a reading nobody can source fails the shapes.

In [8]:
exhibits.show_interface()

confidence = 0.71
  service      : extraction-confidence service (invoice-extraction-cog telemetry, v2.3.1)
  sent payload : {"op":"vendor-fraud-review","run":"run-001","variable":"confidence","batch":"invoice-batch-2026-08"}
  responseCode : 200
  response     : {"confidence":0.71}
consensus_disagreement = 0.10
  service      : consensus-comparator service (v1.4.0)
  sent payload : {"op":"vendor-fraud-review","run":"run-001","variable":"consensus_disagreement","cogs":["invoice-extraction-cog","vendor-risk-cog","anomaly-summary-cog"]}
  responseCode : 200
  response     : {"consensus_disagreement":0.10}
sensitive_data_detected = false
  service      : sensitive-data-scanner service (v5.0.2)
  sent payload : {"op":"vendor-fraud-review","run":"run-001","variable":"sensitive_data_detected","batch":"invoice-batch-2026-08"}
  responseCode : 200
  response     : {"sensitive_data_detected":false}
vendor_risk = high
  service      : vendor-risk-cog scoring endpoint (v0.9.7)
  sent payload : {"